# Lab: Context 전략 비교 시연 (I DO)

같은 태스크(고객 리뷰 감성 분류)를 다양한 컨텍스트 전략으로 실행하고,
토큰 수 / 응답 시간 / 응답 품질을 비교한다.

| 비교 | 설명 |
|------|------|
| 비교 1 | System Prompt 변형 — 없음 vs 역할 지정 vs 역할+제약+출력형식 |
| 비교 2 | 컨텍스트 양 변형 — 최소 컨텍스트 vs 풍부한 컨텍스트 |
| 비교 3 | 구조화 수준 변형 — 자유 응답(텍스트) vs Structured Output(output_type) |

## 환경 설정

PydanticAI와 dotenv를 임포트하고 `.env` 파일에서 API 키를 로드한다.
`time` 모듈은 각 전략의 응답 지연 시간을 측정하는 데 사용한다.

In [1]:
import time
from pydantic import BaseModel, Field
from pydantic_ai import Agent
from dotenv import load_dotenv

# .env 파일에서 OPENAI_API_KEY 등 환경변수를 로드
load_dotenv()

True

## 모델 및 비용 설정

사용할 모델과 토큰당 비용 단가를 정의한다.
비용 계산을 통해 각 전략의 경제성을 비교할 수 있다.

In [4]:
# 사용할 모델 지정
MODEL = "openai:gpt-5.4"

# 비용 단가 ($/1M tokens, 2026년 기준 예시)
# 입력 토큰과 출력 토큰의 단가가 다르다 — 출력이 약 4배 비싸다
PRICING = {
    "openai:gpt-5.4": {"input": 2.50, "output": 10.00},
    "openai:gpt-5.4-mini": {"input": 0.15, "output": 0.60},
}

## 헬퍼 함수

비용 계산, Agent 실행, 결과 출력, 비교 테이블 등 반복되는 로직을 함수로 분리한다.

> **`run_agent()`가 핵심**: `agent.run()`은 비동기(async) 메서드이므로 `await`로 호출한다.
> Jupyter 노트북은 이미 이벤트 루프가 동작 중이라 `run_sync()`를 쓰면 오류가 발생한다.
> 대신 `await agent.run()`을 사용하면 Jupyter의 top-level await 지원 덕분에 정상 동작한다.

In [5]:
def calculate_cost(model: str, input_tokens: int, output_tokens: int) -> float:
    """토큰 사용량 기반 비용 계산 (단위: USD)"""
    prices = PRICING.get(model, {"input": 2.50, "output": 10.00})
    # 단가가 1M 토큰 기준이므로 1_000_000으로 나눈다
    return (input_tokens * prices["input"] + output_tokens * prices["output"]) / 1_000_000


async def run_agent(agent: Agent, prompt: str, label: str) -> dict:
    """Agent를 실행하고 응답·토큰·시간·비용을 딕셔너리로 반환

    Jupyter 노트북은 이벤트 루프가 이미 실행 중이므로
    run_sync() 대신 await agent.run()을 사용한다.
    """
    start = time.time()
    # await로 비동기 실행 — Jupyter에서는 top-level await 가능
    result = await agent.run(prompt)
    elapsed = time.time() - start
    # usage()로 입력/출력 토큰 수를 조회
    usage = result.usage()

    cost = calculate_cost(MODEL, usage.input_tokens, usage.output_tokens)

    return {
        "label": label,
        "output": result.output,
        "input_tokens": usage.input_tokens,
        "output_tokens": usage.output_tokens,
        "total_tokens": usage.input_tokens + usage.output_tokens,
        "latency_sec": round(elapsed, 2),
        "cost_usd": cost,
    }


def print_result(label: str, result: dict):
    """결과를 보기 좋게 출력 — 응답이 길면 120자까지만 표시"""
    output_str = str(result["output"])
    display = output_str[:120] + "..." if len(output_str) > 120 else output_str
    print(f"[{label}]")
    print(f"  응답: {display}")
    print(f"  입력 토큰: {result['input_tokens']}")
    print(f"  출력 토큰: {result['output_tokens']}")
    print(f"  지연:      {result['latency_sec']}초")
    print(f"  비용:      ${result['cost_usd']:.6f}")


def print_comparison_table(results: list[dict]):
    """여러 전략의 결과를 한눈에 비교하는 테이블 출력"""
    print(f"\n{'전략':<30} {'입력토큰':>8} {'출력토큰':>8} {'지연(초)':>8} {'비용($)':>10}")
    print(f"{'─'*70}")
    for r in results:
        print(
            f"{r['label']:<30} {r['input_tokens']:>8} {r['output_tokens']:>8} "
            f"{r['latency_sec']:>8} {r['cost_usd']:>10.6f}"
        )

## 테스트 데이터

동일한 리뷰와 태스크 지시문을 모든 전략에 공통으로 사용한다.
변수를 통일해야 전략 간 비교가 공정해진다.

In [6]:
# 긍정과 부정이 섞인 리뷰 — 분류가 어려운 경계 케이스
REVIEW = "배송이 빠르긴 했는데 포장이 엉망이었어요. 제품은 괜찮은데 다음엔 좀 더 신경 써주세요."

# 모든 전략에 동일한 태스크 지시문 사용
TASK_INSTRUCTION = (
    "다음 고객 리뷰를 분석하여 감성(긍정/부정/중립), "
    "핵심 키워드, 요약을 반환하라."
)

print(f"모델: {MODEL}")
print(f"리뷰: \"{REVIEW}\"")

모델: openai:gpt-5.4
리뷰: "배송이 빠르긴 했는데 포장이 엉망이었어요. 제품은 괜찮은데 다음엔 좀 더 신경 써주세요."


---
## 비교 1: System Prompt 변형

System Prompt(`instructions`)를 어떻게 설정하느냐에 따라 응답 품질과 형식이 달라진다.

| 전략 | System Prompt | 기대 효과 |
|------|--------------|----------|
| A | 없음 | LLM의 일반 지식에만 의존, 형식 불안정 |
| B | 역할 지정 | 전문가 역할 부여로 품질 향상 |
| C | 역할 + 제약 + 출력형식 | JSON 형식 강제, 가장 안정적 |

In [7]:
# 태스크 지시문 + 리뷰를 하나의 User 메시지로 조합
user_msg = f"{TASK_INSTRUCTION}\n\n리뷰: \"{REVIEW}\""
# 3가지 전략의 결과를 모아서 비교할 리스트
compare1_results = []

### 전략 A: System Prompt 없음

`instructions`를 지정하지 않으면 System Prompt가 전송되지 않는다.
LLM은 자신의 일반 지식만으로 태스크를 수행하므로 응답 형식이 불안정하다.

In [8]:
# instructions 없이 Agent 생성 — System Prompt가 비어 있다
agent_a = Agent(MODEL)
# run_agent()는 async 함수이므로 await로 호출
r = await run_agent(agent_a, user_msg, "A. System 없음")
compare1_results.append(r)
print_result("A. System Prompt 없음", r)

[A. System Prompt 없음]
  응답: 감성: 중립

핵심 키워드:
- 빠른 배송
- 포장 불량
- 제품 품질 양호
- 개선 요청

요약:
배송은 빨라서 만족스러웠지만 포장 상태가 좋지 않아 아쉬움이 있었고, 제품 자체는 괜찮았으나 다음에는 포장에 더 신...
  입력 토큰: 76
  출력 토큰: 92
  지연:      2.34초
  비용:      $0.001110


### 전략 B: 역할 지정

`instructions`에 역할을 명시하면 LLM이 해당 전문가처럼 응답한다.
품질은 올라가지만 출력 형식은 여전히 자유롭다.

In [9]:
# instructions에 역할만 부여 — 제약 조건이나 출력 형식은 없다
agent_b = Agent(
    MODEL,
    instructions="당신은 이커머스 고객 경험 분석 전문가입니다.",
)
r = await run_agent(agent_b, user_msg, "B. 역할 지정")
compare1_results.append(r)
print_result("B. System Prompt - 역할 지정", r)

[B. System Prompt - 역할 지정]
  응답: 감성: 중립

핵심 키워드:
- 빠른 배송
- 포장 불량
- 제품 품질 무난
- 개선 요청

요약:
배송은 빨랐지만 포장이 좋지 않았고, 제품 자체는 괜찮았으나 포장 측면의 개선이 필요하다는 의견입니다.
  입력 토큰: 93
  출력 토큰: 76
  지연:      1.6초
  비용:      $0.000992


### 전략 C: 역할 + 제약 + 출력 형식

역할 지정에 더해 구체적인 규칙(JSON만 출력, 감성 3택, 키워드 최대 3개 등)을 명시한다.
제약이 많을수록 입력 토큰은 늘지만 응답 형식이 안정된다.

In [10]:
# instructions에 역할 + 5가지 규칙 + 출력 형식을 모두 명시
agent_c = Agent(
    MODEL,
    instructions=(
        "당신은 이커머스 고객 경험 분석 전문가입니다.\n\n"
        "규칙:\n"
        "1. 반드시 JSON만 출력하라. 다른 텍스트는 금지.\n"
        "2. 감성은 '긍정', '부정', '중립' 중 하나.\n"
        "3. 키워드는 최대 3개.\n"
        "4. 요약은 1문장, 20자 이내.\n"
        "5. 확실하지 않으면 '중립'으로 분류하라."
    ),
)
r = await run_agent(agent_c, user_msg, "C. 역할 + 제약 + 출력형식")
compare1_results.append(r)
print_result("C. System Prompt - 역할 + 제약 + 출력형식", r)

[C. System Prompt - 역할 + 제약 + 출력형식]
  응답: {"감성":"부정","키워드":["빠른 배송","포장 불량","제품 무난"],"요약":"배송 빠르나 포장 엉망"}
  입력 토큰: 177
  출력 토큰: 41
  지연:      1.1초
  비용:      $0.000852


### 비교 1 결과

세 전략의 토큰 수, 지연 시간, 비용을 한눈에 비교한다.
System Prompt가 길어질수록 입력 토큰이 늘지만 응답 품질과 형식 안정성이 올라가는 것을 확인한다.

In [11]:
print_comparison_table(compare1_results)


전략                                 입력토큰     출력토큰    지연(초)      비용($)
──────────────────────────────────────────────────────────────────────
A. System 없음                         76       92     2.34   0.001110
B. 역할 지정                             93       76      1.6   0.000992
C. 역할 + 제약 + 출력형식                   177       41      1.1   0.000852


### 관찰 포인트

- System Prompt가 없으면 응답 형식이 불안정하다
- 역할만 지정하면 품질은 올라가지만 형식이 여전히 자유롭다
- 제약+출력형식을 추가하면 JSON 형식이 안정되지만 토큰이 소폭 증가한다

---
## 비교 2: 컨텍스트 양 변형

같은 태스크라도 LLM에게 제공하는 배경 정보(컨텍스트)의 양에 따라 결과가 달라진다.

| 전략 | 컨텍스트 양 | 트레이드오프 |
|------|-----------|------------|
| A | 최소 (질문만) | 토큰 절약, 품질 불안정 |
| B | 풍부 (배경+기준+예시) | 토큰 증가, 품질 안정 |

In [12]:
# 비교 2 결과를 모을 리스트
compare2_results = []

### 전략 A: 최소 컨텍스트 — 질문만 던짐

System Prompt도 없고 배경 정보도 없다.
LLM이 "분류해줘"라는 모호한 지시만 받으므로 분류 기준이 불명확하다.

In [13]:
# instructions 없이 최소한의 프롬프트만 전달
agent_minimal = Agent(MODEL)
minimal_prompt = f"이 리뷰를 분류해줘: \"{REVIEW}\""

r = await run_agent(agent_minimal, minimal_prompt, "A. 최소 컨텍스트")
compare2_results.append(r)
print_result("A. 최소 컨텍스트 (질문만)", r)

[A. 최소 컨텍스트 (질문만)]
  응답: 분류: 혼합(긍정+부정)

근거:
- 긍정: “배송이 빠르긴 했는데”, “제품은 괜찮은데”
- 부정: “포장이 엉망이었어요”, “다음엔 좀 더 신경 써주세요”

요약:
- 배송 속도와 제품 자체는 긍정적
- 포장 상...
  입력 토큰: 48
  출력 토큰: 120
  지연:      1.99초
  비용:      $0.001320


### 전략 B: 풍부한 컨텍스트 — 배경 정보 + 예시 포함

`instructions`에 회사 배경, 분류 기준(긍정/부정/중립의 정의), Few-shot 예시, 출력 형식을 모두 포함한다.
토큰은 늘지만 일관된 고품질 응답을 기대할 수 있다.

In [14]:
# instructions에 배경 정보 + 분류 기준 + Few-shot 예시 + 출력 형식을 모두 포함
agent_rich = Agent(
    MODEL,
    instructions=(
        "당신은 이커머스 고객 경험 분석 전문가입니다.\n\n"
        "배경: 우리 회사는 전자제품 온라인 쇼핑몰을 운영합니다.\n"
        "리뷰 분류 기준:\n"
        "- 긍정: 재구매 의사, 만족 표현, 추천\n"
        "- 부정: 불만, 클레임, 환불 요청\n"
        "- 중립: 단순 사실 기술, 혼합 감정\n\n"
        "예시:\n"
        "- '최고에요! 꼭 사세요' -> 긍정\n"
        "- '환불 부탁드립니다' -> 부정\n"
        "- '가격 대비 괜찮아요' -> 중립\n\n"
        "반드시 JSON으로 응답하라:\n"
        '{"감성": "...", "키워드": [...], "요약": "...", "신뢰도": 0.0~1.0}'
    ),
)
rich_prompt = f"다음 고객 리뷰를 분석해주세요:\n\n\"{REVIEW}\""

r = await run_agent(agent_rich, rich_prompt, "B. 풍부한 컨텍스트")
compare2_results.append(r)
print_result("B. 풍부한 컨텍스트 (배경+예시 포함)", r)

[B. 풍부한 컨텍스트 (배경+예시 포함)]
  응답: {"감성":"중립","키워드":["배송 빠름","포장 불량","제품 만족","개선 요청"],"요약":"배송은 빨랐고 제품에는 대체로 만족했지만, 포장 상태에 불만이 있어 개선을 요청한 리뷰입니다.","신뢰도":0.9...
  입력 토큰: 218
  출력 토큰: 72
  지연:      1.52초
  비용:      $0.001265


### 비교 2 결과

최소 vs 풍부한 컨텍스트의 토큰/비용/품질 트레이드오프를 확인한다.

In [15]:
print_comparison_table(compare2_results)


전략                                 입력토큰     출력토큰    지연(초)      비용($)
──────────────────────────────────────────────────────────────────────
A. 최소 컨텍스트                           48      120     1.99   0.001320
B. 풍부한 컨텍스트                         218       72     1.52   0.001265


### 관찰 포인트

- 최소 컨텍스트는 토큰이 적지만, 분류 기준이 불명확하여 응답 품질이 불안정
- 풍부한 컨텍스트는 토큰이 늘지만, 배경 정보와 예시 덕분에 일관된 고품질 응답
- 실무에서는 '비용 vs 품질' 트레이드오프를 고려하여 적정 컨텍스트를 설계해야 한다

---
## 비교 3: 구조화 수준 변형

같은 태스크를 자유 텍스트 응답 vs PydanticAI `output_type`(Structured Output)으로 비교한다.

| 전략 | 구조화 수준 | 후속 처리 |
|------|-----------|----------|
| A | 자유 응답 (텍스트) | 파싱 필요, 실패 가능 |
| B | Structured Output | 타입 안전, 바로 사용 가능 |

PydanticAI의 `output_type`에 Pydantic 모델을 지정하면, LLM 응답이 해당 스키마에 100% 맞게 강제된다.

### Structured Output용 Pydantic 모델 정의

`Field(description=...)`은 LLM에게 각 필드의 의미를 알려주는 힌트 역할을 한다.
이 스키마가 `output_type`으로 전달되면 LLM이 JSON Schema를 참조하여 응답한다.

In [ ]:
# Pydantic 모델로 응답 스키마를 정의
# Field의 description은 LLM이 참조하는 힌트 — 명확할수록 결과가 좋다
class ReviewAnalysis(BaseModel):
    """리뷰 분석 결과"""
    sentiment: str = Field(description="감성: '긍정', '부정', '중립' 중 하나")
    keywords: list[str] = Field(description="핵심 키워드 (최대 3개)")
    summary: str = Field(description="리뷰 요약 (1문장)")
    confidence: float = Field(description="분석 신뢰도 (0.0~1.0)", ge=0, le=1)

In [ ]:
user_msg_3 = f"다음 고객 리뷰를 분석해주세요:\n\n\"{REVIEW}\""
compare3_results = []

### 전략 A: 자유 응답 (텍스트)

`output_type`을 지정하지 않으면 LLM이 자유 형식 텍스트로 응답한다.
매번 형식이 달라질 수 있어 후속 코드에서 파싱이 까다롭다.

In [ ]:
# output_type 미지정 — 자유 텍스트 응답
agent_text = Agent(
    MODEL,
    instructions=(
        "고객 리뷰를 분석하여 감성, 키워드, 요약을 알려주세요."
    ),
)
r = await run_agent(agent_text, user_msg_3, "A. 자유 응답 (텍스트)")
compare3_results.append(r)
print_result("A. 자유 응답 (텍스트)", r)

### 전략 B: Structured Output

PydanticAI의 `output_type=ReviewAnalysis`로 스키마를 강제한다.
`result.output`이 `ReviewAnalysis` 인스턴스가 되어 `.sentiment`, `.keywords` 등에 타입 안전하게 접근 가능하다.

In [ ]:
# output_type에 Pydantic 모델을 지정 — 응답이 스키마에 맞게 강제된다
agent_structured = Agent(
    MODEL,
    instructions=(
        "고객 리뷰를 분석하여 구조화된 결과를 반환하세요."
    ),
    output_type=ReviewAnalysis,
)
r = await run_agent(agent_structured, user_msg_3, "B. Structured Output")
compare3_results.append(r)
print_result("B. Structured Output (output_type)", r)

### Structured Output 상세 확인

`result.output`이 `ReviewAnalysis` 인스턴스인지 확인하고 각 필드에 접근한다.
이것이 Structured Output의 핵심 장점 — 후속 코드에서 파싱 없이 바로 사용 가능하다.

In [ ]:
# Structured Output이면 Pydantic 모델 필드에 직접 접근 가능
structured_output = r["output"]
if isinstance(structured_output, ReviewAnalysis):
    print(f"감성:   {structured_output.sentiment}")
    print(f"키워드: {', '.join(structured_output.keywords)}")
    print(f"요약:   {structured_output.summary}")
    print(f"신뢰도: {structured_output.confidence:.0%}")
else:
    print(f"[주의] output_type이 적용되지 않음: {type(structured_output)}")

### 비교 3 결과

자유 응답 vs Structured Output의 토큰/비용 차이와 형식 안정성을 비교한다.

In [ ]:
print_comparison_table(compare3_results)

### 관찰 포인트

- 자유 응답은 형식이 매번 달라질 수 있어 파싱이 어렵다
- Structured Output(output_type)은 Pydantic 모델로 응답을 100% 보장한다
- 후속 코드에서 `result.output.sentiment`처럼 타입 안전하게 접근 가능
- Agent 파이프라인에서는 Structured Output이 필수 — 다음 단계 입력으로 바로 사용

---
## 최종 요약: Context 전략 체크리스트

**1. System Prompt (instructions)**
- 역할 + 제약 조건을 명시하라
- 출력 형식을 강제하면 파싱 안정성이 올라간다
- PydanticAI에서는 Agent의 instructions 파라미터로 설정

**2. 컨텍스트 양**
- 최소 컨텍스트: 토큰 절약, 품질 불안정
- 풍부한 컨텍스트: 토큰 증가, 품질 안정
- 배경 정보 + 분류 기준 + 예시 조합이 효과적

**3. 구조화 수준**
- 자유 응답: 유연하지만 파싱 불안정
- Structured Output: output_type으로 스키마 강제, 타입 안전
- Agent 파이프라인에서는 Structured Output이 필수